In [1]:
import pandas as pd
import numpy as np
import docplex.cp.model as cp
import itertools
from itertools import combinations
import random
import math
import pickle
import matplotlib.pyplot as plt
import docplex

# CPLEX Optimization Studio is required
ctx = docplex.cp.config.get_default()
ctx['log_output'] = None
ctx['add_log_to_solution'] = False

from docplex.mp.model import Model


In [2]:
from docplex.cp.model import CpoModel
from itertools import product
from typing import Dict, Any, List

def create_robust_scheduling_model(data: Dict[str, Any]) -> CpoModel:

    mdl = CpoModel()

    J: List[int] = data["J"]
    K: List[str] = data["K"]
    M_o: Dict[int, List[int]] = data["M_o"]
    route_ops: Dict[int, List[int]] = data["route_ops"]
    fixed_route_j: Dict[int, int] = data["fixed_route_j"]
    p_jorm: Dict[tuple, int] = data["p_jorm"]   # ⭐ job & route dependent

    Operations: List[tuple] = []
    for j in J:
        r = fixed_route_j.get(j)
        if r is None:
            continue
        for o in route_ops.get(r, []):
            Operations.append((o, j))

    o_first = {j: route_ops[fixed_route_j[j]][0] for j in J}
    o_last  = {j: route_ops[fixed_route_j[j]][-1] for j in J}

    op_int = {
        (o, j): mdl.interval_var(name=f"Op_{o}_{j}")
        for (o, j) in Operations
    }

    x = {}
    machine_activities: Dict[int, List] = {}

    for (o, j) in Operations:
        r = fixed_route_j[j]
        alternatives = []

        for m in M_o.get(o, []):
            x[(o, j, m)] = mdl.binary_var(name=f"X_{o}_{j}_{m}")

            duration = p_jorm.get((j, o, r, m), 0)

            act = mdl.interval_var(
                size=duration,
                optional=True,
                name=f"Act_{o}_{j}_{r}_{m}"
            )

            mdl.add(mdl.presence_of(act) == x[(o, j, m)])

            alternatives.append(act)
            machine_activities.setdefault(m, []).append(act)

        if not alternatives:
            raise ValueError(
                f"No machine alternatives for operation {o}, job {j}, route {r}"
            )

        mdl.add(mdl.alternative(op_int[(o, j)], alternatives))
        mdl.add(mdl.sum(x[(o, j, m)] for m in M_o.get(o, [])) == 1)

    C_max = mdl.max(mdl.end_of(op_int[(o, j)]) for (o, j) in Operations)
    mdl.add(mdl.minimize(C_max))

    for acts in machine_activities.values():
        if acts:
            mdl.add(mdl.no_overlap(acts))

    for j in J:
        r = fixed_route_j[j]
        ops = route_ops[r]
        for i in range(len(ops) - 1):
            mdl.add(
                mdl.end_before_start(
                    op_int[(ops[i], j)],
                    op_int[(ops[i + 1], j)]
                )
            )

    z = {(j, jp, k): mdl.integer_var(min=0, name=f"Z_{j}_{jp}_{k}")
         for j, jp, k in product(J, J, K)}

    w = {(j, jp, k): mdl.binary_var(name=f"W_{j}_{jp}_{k}")
         for j, jp, k in product(J, J, K)}

    for j, k in product(J, K):
        mdl.add(z[(j, j, k)] == 0)
        mdl.add(w[(j, j, k)] == 0)


    for j, k in product(J, K):
        r = fixed_route_j[j]

        demand = data.get("a_jrk", {}).get((j, r, k), 0)
        mdl.add(mdl.sum(z[(jp, j, k)] for jp in J) == demand)

        supply_cap = data.get("b_underline_jrk", {}).get((j, r, k), 0)
        mdl.add(mdl.sum(z[(j, jp, k)] for jp in J) <= supply_cap)

    U = data.get("U_jk", {})
    BIG_M = 100000

    for j, jp, k in product(J, J, K):
        if j != jp:
            mdl.add(z[(j, jp, k)] <= BIG_M * w[(j, jp, k)])
            mdl.add(mdl.if_then(z[(j, jp, k)] >= 1, w[(j, jp, k)] == 1))

    for j, jp, k in product(J, J, K):
        if j != jp:
            mdl.add(
                mdl.if_then(
                    w[(j, jp, k)] == 1,
                    mdl.end_of(op_int[(o_last[j], j)])
                    <= mdl.start_of(op_int[(o_first[jp], jp)])
                )
            )

    return mdl


In [3]:
from itertools import product
from typing import List, Dict, Optional
import random

TOTAL_MODULES = 16
MODULE_TIME = 2

J = list(range(1, 6))       # Jobs 1–5
K = ['G', 'D', 'B']
O_set = set(range(1, 21))
R = [1, 2, 3, 4, 5]
M = [1, 2, 3, 4, 5]
H = 100000

GRADE_MODULE_RANGES = {
    'A': (12, 15),
    'B': (6, 11),
    'C': (0, 5),
}

M_o = {
    1: [1, 2],  2: [3],     3: [1, 2],  4: [1, 2],
    5: [1, 2],  6: [1, 2],  7: [5],     8: [4],
    9: [1, 2],  10: [5],    11: [1, 2], 12: [1, 2],
    13: [3],    14: [1, 2], 15: [4],    16: [3],
    17: [4],    18: [3, 5], 19: [1, 2], 20: [1, 2]
}

route_ops = {
    1: [1, 2, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17],
    2: [1, 2, 4, 5, 8, 12, 13, 15, 16, 17],
    3: [1, 2, 3, 4, 5, 6, 7, 8],
    4: [9, 10, 11, 12, 13, 14, 15, 16, 17],
    5: [1, 2, 3, 14, 15, 16, 17],
}

BASE_OP_TIMES = {
    1: 3, 2: 5, 3: 12, 4: 4, 5: 11,
    6: 19, 8: 20, 9: 17, 11: 16,
    12: 18, 13: 4, 14: 11, 15: 8,
    16: 9, 17: 15, 18: 6, 19: 23, 20: 12
}

def generate_battery_data(
    grades: List[str],
    routes: Optional[List[int]] = None
) -> Dict:

    num_jobs = len(grades)
    jobs = list(range(1, num_jobs + 1))

    # Sample GOOD modules according to grade
    GOOD_MODULES = {}
    for j, g in zip(jobs, grades):
        low, high = GRADE_MODULE_RANGES[g]
        GOOD_MODULES[j] = high
    DEGRADED_MODULES = {j: TOTAL_MODULES - GOOD_MODULES[j] for j in jobs}

    R_j = {j: R for j in jobs}
    O_j = {j: list(O_set) for j in jobs}
    delta_or = {(o, r): int(o in route_ops[r]) for r in R for o in O_set}

    p_jorm = {}
    for j, o, r, m in product(jobs, O_set, R, M):
        if m not in M_o.get(o, []):
            continue

        if o == 7:  # Remove modules
            p_jorm[(j, o, r, m)] = (
                MODULE_TIME * DEGRADED_MODULES[j] if r == 1 else
                MODULE_TIME * TOTAL_MODULES if r == 3 else 0
            )
        elif o == 10:  # Place modules
            p_jorm[(j, o, r, m)] = (
                MODULE_TIME * DEGRADED_MODULES[j] if r == 1 else
                MODULE_TIME * TOTAL_MODULES if r == 4 else 0
            )
        else:
            p_jorm[(j, o, r, m)] = BASE_OP_TIMES.get(o, 0)

    a_jrk = {}
    for j in jobs:
        a_jrk[(j, 1, 'G')] = DEGRADED_MODULES[j]  # repair
        a_jrk[(j, 2, 'B')] = 0                    # BMC replace
        a_jrk[(j, 4, 'G')] = TOTAL_MODULES        # reassembly
        a_jrk[(j, 4, 'B')] = 1
        a_jrk[(j, 5, 'B')] = 0                    # energy storage

    b_underline_jrk = {(j, 3, 'G'): GOOD_MODULES[j] for j in jobs}
    b_underline_jrk.update({(j, 3, 'D'): DEGRADED_MODULES[j] for j in jobs})
    b_underline_jrk.update({(j, 3, 'B'): 1 for j in jobs})

    U_jk = {(j, 'G'): GOOD_MODULES[j] for j in jobs}
    U_jk.update({(j, 'D'): DEGRADED_MODULES[j] for j in jobs})
    U_jk.update({(j, 'B'): 1 for j in jobs})

    if routes is not None:
        fixed_route_j = {j: routes[i] for i, j in enumerate(jobs)}
    else:
        fixed_route_j = {j: random.choice(R) if random.random() < 0.4 else 3 for j in jobs}

    data = {
        'J': jobs,
        'O_j': O_j,
        'R_j': R_j,
        'M_o': M_o,
        'K': K,
        'p_jorm': p_jorm,
        'delta_or': delta_or,
        'a_jrk': a_jrk,
        'b_underline_jrk': b_underline_jrk,
        'U_jk': U_jk,
        'H': H,
        'route_ops': route_ops,
        'fixed_route_j': fixed_route_j,
        'GOOD_MODULES': GOOD_MODULES,
        'DEGRADED_MODULES': DEGRADED_MODULES
    }

    return data

def test_generate_battery_data(
    grades: List[str],
    routes: Optional[List[int]] = None
) -> Dict:

    num_jobs = len(grades)
    jobs = list(range(1, num_jobs + 1))

    GOOD_MODULES = {}
    for j, g in zip(jobs, grades):
        low, high = GRADE_MODULE_RANGES[g]
        GOOD_MODULES[j] = random.randint(low, high)
    DEGRADED_MODULES = {j: TOTAL_MODULES - GOOD_MODULES[j] for j in jobs}

    R_j = {j: R for j in jobs}
    O_j = {j: list(O_set) for j in jobs}
    delta_or = {(o, r): int(o in route_ops[r]) for r in R for o in O_set}

    p_jorm = {}
    for j, o, r, m in product(jobs, O_set, R, M):
        if m not in M_o.get(o, []):
            continue

        if o == 7:  # Remove modules
            p_jorm[(j, o, r, m)] = (
                MODULE_TIME * DEGRADED_MODULES[j] if r == 1 else
                MODULE_TIME * TOTAL_MODULES if r == 3 else 0
            )
        elif o == 10:  # Place modules
            p_jorm[(j, o, r, m)] = (
                MODULE_TIME * DEGRADED_MODULES[j] if r == 1 else
                MODULE_TIME * TOTAL_MODULES if r == 4 else 0
            )
        else:
            p_jorm[(j, o, r, m)] = BASE_OP_TIMES.get(o, 0)

    a_jrk = {}
    for j in jobs:
        a_jrk[(j, 1, 'G')] = DEGRADED_MODULES[j]  # repair
        a_jrk[(j, 2, 'B')] = 0                    # BMC replace
        a_jrk[(j, 4, 'G')] = TOTAL_MODULES        # reassembly
        a_jrk[(j, 4, 'B')] = 1
        a_jrk[(j, 5, 'B')] = 0                    # energy storage

    b_underline_jrk = {(j, 3, 'G'): GOOD_MODULES[j] for j in jobs}
    b_underline_jrk.update({(j, 3, 'D'): DEGRADED_MODULES[j] for j in jobs})
    b_underline_jrk.update({(j, 3, 'B'): 1 for j in jobs})

    U_jk = {(j, 'G'): GOOD_MODULES[j] for j in jobs}
    U_jk.update({(j, 'D'): DEGRADED_MODULES[j] for j in jobs})
    U_jk.update({(j, 'B'): 1 for j in jobs})

    if routes is not None:
        fixed_route_j = {j: routes[i] for i, j in enumerate(jobs)}
    else:
        fixed_route_j = {j: random.choice(R) if random.random() < 0.4 else 3 for j in jobs}

    data = {
        'J': jobs,
        'O_j': O_j,
        'R_j': R_j,
        'M_o': M_o,
        'K': K,
        'p_jorm': p_jorm,
        'delta_or': delta_or,
        'a_jrk': a_jrk,
        'b_underline_jrk': b_underline_jrk,
        'U_jk': U_jk,
        'H': H,
        'route_ops': route_ops,
        'fixed_route_j': fixed_route_j,
        'GOOD_MODULES': GOOD_MODULES,
        'DEGRADED_MODULES': DEGRADED_MODULES
    }

    return data

In [4]:
import random
import numpy as np
# Removed: import multiprocessing (for serial execution)
from deap import base, creator, tools, algorithms

# Define the necessary global variables and initial parameters
grades = ['A', 'A', 'B', 'B', 'B', 'B', 'C']

# Route values differentiated by grade
ROUTE_VALUE_BY_GRADE = {
    'A': {
        1: 200,   # full restoration
        2: 100,   # BMC rebalance
        3: 0,     # no action / discard
        4: 200,   # full restoration
        5: 75    # bypass damaged cell
    },
    'B': {
        1: 200,
        2: 70,
        3: 0,
        4: 200,
        5: 35
    },
    'C': {
        1: 200,
        2: 40,
        3: 0,
        4: 200,
        5: 20
    }
}


NUM_JOBS = len(grades)  
ROUTES = sorted({
    r
    for grade_dict in ROUTE_VALUE_BY_GRADE.values()
    for r in grade_dict.keys()
})

# --- Evaluation Function ---
def evaluate_solution(route_selection):
    
    makespan = 1000  # fallback penalty
    value = 0        # fallback value

    try:
        # Build stochastic instance
        data = generate_battery_data(grades, route_selection)
        mdl = create_robust_scheduling_model(data)
        sol = mdl.solve(TimeLimit=0.1)

        # -------------------------------
        # Module-change calculation
        # -------------------------------
        count_r3 = route_selection.count(3)
        count_r4 = route_selection.count(4)
        total_remaining = (16 * count_r3) - (16 * count_r4)

        # -------------------------------
        # Extract solution
        # -------------------------------
        if sol is not None and sol.get_objective_values() is not None:
            makespan = sol.get_objective_values()[0]

            # Grade-aware route value
            base_value = sum(
                ROUTE_VALUE_BY_GRADE[g][r]
                for g, r in zip(grades, route_selection)
            )

            # Module batching bonus / penalty
            value = base_value + np.floor(total_remaining) * 3

    except Exception:
        pass

    return makespan, -value

def test_evaluate_solution(route_selection):
    
    makespan = 1000  # fallback penalty
    value = 0        # fallback value

    try:
        # Build stochastic instance
        data = test_generate_battery_data(grades, route_selection)
        mdl = create_robust_scheduling_model(data)
        sol = mdl.solve(TimeLimit=0.1)

        # -------------------------------
        # Module-change calculation
        # -------------------------------
        count_r3 = route_selection.count(3)
        count_r4 = route_selection.count(4)
        total_remaining = (16 * count_r3) - (16 * count_r4)

        # -------------------------------
        # Extract solution
        # -------------------------------
        if sol is not None and sol.get_objective_values() is not None:
            makespan = sol.get_objective_values()[0]

            # Grade-aware route value
            base_value = sum(
                ROUTE_VALUE_BY_GRADE[g][r]
                for g, r in zip(grades, route_selection)
            )

            # Module batching bonus / penalty
            value = base_value + np.floor(total_remaining) * 3

    except Exception:
        pass

    return makespan, -value

In [5]:
creator.create("FitnessMulti", base.Fitness, weights=(-1.0, -1.0))
creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()

toolbox.register("attr_route", random.choice, ROUTES)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_route, n=NUM_JOBS)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evaluate_solution)
toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", tools.mutUniformInt, low=min(ROUTES), up=max(ROUTES), indpb=0.2)
toolbox.register("select", tools.selNSGA2)

toolbox.register("map", map) 
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("min_mspan", np.min, axis=0)
stats.register("avg_mspan", np.mean, axis=0)
stats.register("max_val", lambda values: np.max(-np.array(values), axis=0))
stats.register("avg_val", lambda values: np.mean(-np.array(values), axis=0))

POP_SIZE = 50
NGEN = 40
CXPB = 0.7
MUTPB = 0.2

print(f"Starting NSGA-II evolution for {NGEN} generations with {POP_SIZE} individuals...")
pop = toolbox.population(n=POP_SIZE)

pop, logbook = algorithms.eaMuPlusLambda(pop, toolbox, mu=POP_SIZE, lambda_=POP_SIZE, 
                                         cxpb=CXPB, mutpb=MUTPB, ngen=NGEN, 
                                         stats=stats, halloffame=None, verbose=True)

print("\nEvolution finished.")
print("\n--- Intermediate Steps (Statistics per Generation) ---")
header = "{:^5} | {:^10} | {:^10} | {:^10} | {:^10}".format("Gen", "Min_Mspan", "Avg_Mspan", "Max_Value", "Avg_Value")
print(header)
print("-" * len(header))

for gen in logbook:
    print("{:^5} | {:^10.2f} | {:^10.2f} | {:^10.2f} | {:^10.2f}".format(
        gen['gen'], 
        gen['min_mspan'][0], 
        gen['avg_mspan'][0], 
        -gen['max_val'][1], 
        -gen['avg_val'][1]))

# Extract Pareto front
pareto_front = tools.sortNondominated(pop, k=len(pop), first_front_only=True)[0]
pareto_solutions = [(ind, ind.fitness.values) for ind in pareto_front]

print("\n--- Final Pareto Front Solutions ---")
print(f"Found {len(pareto_solutions)} non-dominated solutions.")

for i, (sol, obj) in enumerate(pareto_solutions):
    print(f"Solution {i+1}: Routes: {sol} | Makespan: {obj[0]:.2f} | Value: {-obj[1]:.2f}")
    if i >= 4:
        print("...")
        break

Starting NSGA-II evolution for 40 generations with 50 individuals...
gen	nevals	min_mspan    	avg_mspan        	max_val      	avg_val          
0  	50    	[ 206. -871.]	[ 758.56 -203.16]	[-206.  871.]	[-758.56  203.16]
1  	43    	[ 206. -971.]	[ 560.7  -381.86]	[-206.  971.]	[-560.7   381.86]
2  	49    	[ 206. -971.]	[ 285.86 -603.08]	[-206.  971.]	[-285.86  603.08]
3  	43    	[ 206. -971.]	[ 274.44 -608.8 ]	[-206.  971.]	[-274.44  608.8 ]
4  	45    	[ 195. -971.]	[ 273.06 -627.16]	[-195.  971.]	[-273.06  627.16]
5  	45    	[ 195. -971.]	[ 269.34 -616.58]	[-195.  971.]	[-269.34  616.58]
6  	47    	[ 195. -971.]	[ 267.48 -611.48]	[-195.  971.]	[-267.48  611.48]
7  	46    	[ 195. -971.]	[ 270.02 -626.22]	[-195.  971.]	[-270.02  626.22]
8  	41    	[ 195. -971.]	[ 268.64 -623.56]	[-195.  971.]	[-268.64  623.56]
9  	41    	[ 195. -971.]	[ 267.92 -622.1 ]	[-195.  971.]	[-267.92  622.1 ]
10 	42    	[ 195. -971.]	[ 264.3 -607.5]  	[-195.  971.]	[-264.3  607.5]  
11 	44    	[ 195. -996.]	[ 265.

In [ ]:
pareto_front = tools.sortNondominated(pop, k=len(pop), first_front_only=True)[0]
pareto_solutions = [(ind, ind.fitness.values) for ind in pareto_front]

makespan_values = [ind.fitness.values[0] for ind in pareto_front]
value_values = [-ind.fitness.values[1] for ind in pareto_front]

plt.figure(figsize=(10, 6))
plt.scatter(makespan_values, value_values, marker='o', color='blue', label='Pareto Front Solutions')
plt.title('Pareto Front Visualization (Makespan vs. Value)')
plt.xlabel('Makespan (Objective 1, Minimization)')
plt.ylabel('Total Value (Objective 2, Maximization)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\n--- Final Pareto Front Solutions ---")
print(f"Found {len(pareto_solutions)} non-dominated solutions.")

for i, (sol, obj) in enumerate(pareto_solutions):
    print(f"Solution {i+1}: Routes: {sol} | Makespan: {obj[0]:.2f} | Value: {-obj[1]:.2f}")
    if i >= 104:
        print("...")
        break